In [1]:
! pip install ultralytics roboflow opencv-python huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.0 MB/s eta 0:00:00:00:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

In [2]:
import os
import json
from ultralytics import YOLO
from roboflow import Roboflow
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
HF_TOKEN      = UserSecretsClient().get_secret("HF_TOKEN")
api           = HfApi(token=HF_TOKEN)
HF_REPO_ID    = f"{api.whoami()['name']}/Smoking-detection-YOLO26s"
STATUS_FILE = "training_status.json"
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)
print(f"[HF] Repo → https://huggingface.co/{HF_REPO_ID}")

[HF] Repo → https://huggingface.co/basant18/Smoking-detection-YOLO26s


In [5]:
rf_api_key = UserSecretsClient().get_secret("RF")
rf = Roboflow(api_key=rf_api_key)
project = rf.workspace("pro-hiyaw").project("cigarettes-reality-2-trnm2")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Cigarettes-reality-2-1 in yolo26:: 100%|██████████| 15981/15981 [00:02<00:00, 6718.51it/s] 


In [6]:
# 2. Check for existing training status to Resume
resume_training = True
model_weights = 'yolo26s.pt' # Default starting weights

In [7]:
if os.path.exists(STATUS_FILE):
    with open(STATUS_FILE, "r") as f:
        status = json.load(f)
        
    # If it crashed previously, load the last saved weights
    if status.get("status") == "training" and os.path.exists(status.get("weights_path")):
        print(f"Crash detected! Resuming training from epoch {status['last_epoch']}...")
        model_weights = status["weights_path"]
        resume_training = True

In [8]:
# Initialize Model
model = YOLO(model_weights)

In [9]:
# 3. Define the Callback Function
def on_train_epoch_end(trainer):
    """Callback triggered at the end of every epoch."""
    epoch = trainer.epoch + 1  # trainer.epoch is 0-indexed
    weights_path = str(trainer.last) # Path to runs/detect/train/weights/last.pt
    best_weights_path = str(trainer.best) # Path to runs/detect/train/weights/best.pt
    
    # --- Save JSON Status ---
    status_data = {
        "last_epoch": epoch,
        "weights_path": weights_path,
        "best_weights_path": best_weights_path,
        "status": "training"
    }
    with open(STATUS_FILE, "w") as f:
        json.dump(status_data, f)
        
    # --- Upload to Hugging Face every 10 epochs ---
    if epoch % 10 == 0:
        print(f"\n--- Epoch {epoch}: Uploading checkpoints to Hugging Face ---")
        try:
            # Upload last.pt (Checkpoint to resume from)
            api.upload_file(
                path_or_fileobj=weights_path,
                path_in_repo=f"weights/epoch_{epoch}_last.pt",
                repo_id=HF_REPO_ID,
                token=HF_TOKEN,
                repo_type="model"
            )
            
            # Upload best.pt (Best performing model so far)
            if os.path.exists(best_weights_path):
                api.upload_file(
                    path_or_fileobj=best_weights_path,
                    path_in_repo="weights/best.pt", # Overwrites the best.pt in the repo
                    repo_id=HF_REPO_ID,
                    token=HF_TOKEN,
                    repo_type="model"
                )
                
            # Upload the JSON status file
            api.upload_file(
                path_or_fileobj=STATUS_FILE,
                path_in_repo=STATUS_FILE,
                repo_id=HF_REPO_ID,
                token=HF_TOKEN,
                repo_type="model"
            )
            print("Upload successful!\n")
        except Exception as e:
            print(f"Failed to upload to Hugging Face: {e}\n")


In [10]:
# Attach the callback to the model
model.add_callback("on_train_epoch_end", on_train_epoch_end)

In [11]:
# 4. Train the model
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=200,          
    imgsz=800,          
    batch=16,           
    device=0,
    patience=30,        # <--- EARLY STOPPING: Stops if no improvement after 25 epochs
    resume=resume_training, 
    hsv_s=0.2,           # Vary saturation (lighting changes)
    hsv_v=0.2,           # Vary exposure (lighting changes)
    degrees=10.0,        # Rotate images slightly (head angles vary)
    fliplr=0.5           # Flip left/right# <--- RESUME: True if JSON found a crash
)

WARNING ⚠️ model 'yolo26s.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Cigarettes-reality-2-1/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.2, hsv_v=0.2, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Upload successful!

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.7it/s 3.7s.4ss
                   all        307        295      0.889      0.888      0.896      0.527

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    111/200      7.58G      1.038     0.6616    0.01036         11        800: 100% ━━━━━━━━━━━━ 461/461 1.6it/s 4:42<0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.7it/s 3.6s.4ss
                   all        307        295      0.896      0.885      0.893      0.528

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    112/200      7.57G       1.03     0.6597    0.01054         13        800: 100% ━━━━━━━━━━━━ 461/461 1.6it/s 4:42<0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.7it/s 3.6s.4ss
     

In [11]:
# 5. Clean up status on successful completion
if os.path.exists(STATUS_FILE):
    with open(STATUS_FILE, "w") as f:
        json.dump({"status": "completed", "final_weights": str(model.trainer.best)}, f)
    
print("Training completed successfully!")

Training completed successfully!


In [1]:
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image

# ══════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════

MODEL_PATH  = '/kaggle/input/models/basanttyasser/smoking-model/keras/default/1/best_model.keras'
IMG_SIZE    = (224, 224)
CLASS_NAMES = ['Not Smoking', 'Smoking']
THRESHOLD   = 0.5
CAM_INDEX   = 0       # change to 1, 2 … if wrong camera

# Colors (BGR for OpenCV)
COLOR_SMOKING     = (73,  81, 248)   # red
COLOR_NOT_SMOKING = (80, 185,  63)   # green
COLOR_WARN        = (34, 153, 209)   # amber
COLOR_BG          = (23,  27,  33)   # dark

# ══════════════════════════════════════════════════════════
#  LOAD MODEL
# ══════════════════════════════════════════════════════════

print('Loading model …')
model = tf.keras.models.load_model(MODEL_PATH)
print('Model ready.  Press  Q  to quit.\n')

# ══════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════

def preprocess(frame):
    img   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img   = cv2.resize(img, IMG_SIZE)
    img   = img.astype('float32') / 255.0
    return np.expand_dims(img, axis=0)


def draw_bar(frame, prob_smoking, x, y, w, h, color):
    """Draws a filled probability bar."""
    filled = int(w * prob_smoking)
    cv2.rectangle(frame, (x, y), (x + w, y + h), (50, 50, 60), -1)       # bg
    cv2.rectangle(frame, (x, y), (x + filled, y + h), color, -1)          # fill
    cv2.rectangle(frame, (x, y), (x + w, y + h), (100, 100, 110), 1)      # border


def draw_overlay(frame, label, confidence, prob_smoking):
    h, w = frame.shape[:2]
    is_smoking = label == 'Smoking'
    color      = COLOR_SMOKING if is_smoking else COLOR_NOT_SMOKING

    # ── Top banner ────────────────────────────────────────
    cv2.rectangle(frame, (0, 0), (w, 60), COLOR_BG, -1)
    cv2.putText(frame, 'Smoking Detector',
                (12, 40), cv2.FONT_HERSHEY_DUPLEX,
                1.1, (200, 200, 200), 1, cv2.LINE_AA)

    # ── Prediction badge ──────────────────────────────────
    badge_x, badge_y = w - 320, 10
    cv2.rectangle(frame,
                  (badge_x - 10, badge_y),
                  (w - 10, badge_y + 44),
                  color, -1)
    cv2.putText(frame,
                f'{label}  {confidence:.0%}',
                (badge_x, badge_y + 31),
                cv2.FONT_HERSHEY_DUPLEX,
                0.85, (255, 255, 255), 1, cv2.LINE_AA)

    # ── Bottom HUD ────────────────────────────────────────
    hud_y = h - 80
    cv2.rectangle(frame, (0, hud_y - 10), (w, h), COLOR_BG, -1)

    # Not Smoking bar
    cv2.putText(frame, 'Not Smoking',
                (12, hud_y + 18),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55, COLOR_NOT_SMOKING, 1, cv2.LINE_AA)
    draw_bar(frame, 1 - prob_smoking,
             130, hud_y + 4, 200, 18, COLOR_NOT_SMOKING)
    cv2.putText(frame, f'{1-prob_smoking:.0%}',
                (336, hud_y + 18),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55, COLOR_NOT_SMOKING, 1, cv2.LINE_AA)

    # Smoking bar
    cv2.putText(frame, 'Smoking',
                (12, hud_y + 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55, COLOR_SMOKING, 1, cv2.LINE_AA)
    draw_bar(frame, prob_smoking,
             130, hud_y + 36, 200, 18, COLOR_SMOKING)
    cv2.putText(frame, f'{prob_smoking:.0%}',
                (336, hud_y + 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55, COLOR_SMOKING, 1, cv2.LINE_AA)

    # ── Border flash when smoking ──────────────────────────
    if is_smoking and confidence > 0.75:
        thickness = 6
        cv2.rectangle(frame, (0, 0), (w - 1, h - 1), COLOR_SMOKING, thickness)

    return frame


# ══════════════════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════════════════

def run():
    cap = cv2.VideoCapture(CAM_INDEX)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open camera index {CAM_INDEX}')

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # Run inference every N frames to keep it smooth
    INFER_EVERY = 5
    frame_count  = 0
    label        = '…'
    confidence   = 0.0
    prob_smoking = 0.5

    while True:
        ret, frame = cap.read()
        if not ret:
            print('Camera read failed.')
            break

        frame_count += 1

        if frame_count % INFER_EVERY == 0:
            tensor       = preprocess(frame)
            prob_smoking = float(model.predict(tensor, verbose=0)[0][0])
            idx          = int(prob_smoking >= THRESHOLD)
            label        = CLASS_NAMES[idx]
            confidence   = prob_smoking if idx == 1 else 1 - prob_smoking

        draw_overlay(frame, label, confidence, prob_smoking)
        cv2.imshow('Smoking Detector  —  press Q to quit', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print('Done.')


if __name__ == '__main__':
    run()

2026-04-16 16:09:38.856526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776355779.073618      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776355779.144161      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776355779.687439      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776355779.687489      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776355779.687492      55 computation_placer.cc:177] computation placer alr

Loading model …


I0000 00:00:1776355808.473247      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776355808.479316      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model ready.  Press  Q  to quit.



[ WARN:0@34.623] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@34.649] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 0
> must be less than
>     'device_list->nb_devices' is 0


[ERROR:0@34.650] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


RuntimeError: Cannot open camera index 0